# 交互式 FFT 滤波演示

使用 ipywidgets 实现交互式频域滤波：
- 低通滤波：保留低频成分，去除高频
- 高通滤波：保留高频成分，去除低频
- 带通滤波：保留指定频率范围
- 实时显示滤波后的时域信号
- 导出滤波后数据

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider, Dropdown, Button, VBox, HBox, Output
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. 生成测试信号

In [ ]:
np.random.seed(42)

sampling_freq = 1000  # 采样频率
duration = 2          # 信号时长
t = np.linspace(0, duration, int(sampling_freq * duration), endpoint=False)
N = len(t)

# 生成多频率成分信号
freq_components = [5, 15, 30, 60, 120]  # 频率成分
amplitudes = [1.0, 0.8, 0.5, 0.3, 0.2]   # 对应幅度

signal = np.zeros_like(t)
for freq, amp in zip(freq_components, amplitudes):
    signal += amp * np.sin(2 * np.pi * freq * t)

# 添加噪声
noise_level = 0.15
noise = noise_level * np.random.randn(len(t))
original_signal = signal + noise

# 计算原始信号的 FFT
fft_original = np.fft.fft(original_signal)
frequencies = np.fft.fftfreq(N, 1/sampling_freq)

print(f"信号包含的频率成分: {freq_components} Hz")
print(f"采样频率: {sampling_freq} Hz")
print(f"奈奎斯特频率: {sampling_freq/2} Hz")

## 2. 交互式滤波控件

In [ ]:
filtered_signal_global = None

# 滤波类型选择
filter_type = Dropdown(
    options=['低通滤波', '高通滤波', '带通滤波'],
    value='低通滤波',
    description='滤波类型:',
    style={'description_width': 'initial'}
)

# 低通/高通截止频率
cutoff_freq = FloatSlider(
    value=50,
    min=1,
    max=sampling_freq/2,
    step=1,
    description='截止频率 (Hz):',
    style={'description_width': 'initial'},
    layout={'width': '500px'}
)

# 带通滤波下限
low_cutoff = FloatSlider(
    value=20,
    min=1,
    max=sampling_freq/2 - 10,
    step=1,
    description='下限频率 (Hz):',
    style={'description_width': 'initial'},
    layout={'width': '500px'}
)

# 带通滤波上限
high_cutoff = FloatSlider(
    value=80,
    min=10,
    max=sampling_freq/2,
    step=1,
    description='上限频率 (Hz):',
    style={'description_width': 'initial'},
    layout={'width': '500px'}
)

# 显示范围
time_range = FloatSlider(
    value=0.5,
    min=0.1,
    max=duration,
    step=0.1,
    description='显示时长 (s):',
    style={'description_width': 'initial'},
    layout={'width': '500px'}
)

In [ ]:
def apply_filter(fft_data, freqs, filter_mode, cutoff=None, low_cut=None, high_cut=None):
    """
    应用频域滤波
    """
    fft_filtered = fft_data.copy()
    
    if filter_mode == '低通滤波':
        # 保留低于截止频率的成分
        mask = np.abs(freqs) > cutoff
        fft_filtered[mask] = 0
        
    elif filter_mode == '高通滤波':
        # 保留高于截止频率的成分
        mask = np.abs(freqs) < cutoff
        fft_filtered[mask] = 0
        
    elif filter_mode == '带通滤波':
        # 保留指定频率范围内的成分
        mask = (np.abs(freqs) < low_cut) | (np.abs(freqs) > high_cut)
        fft_filtered[mask] = 0
    
    return fft_filtered

In [ ]:
output_plot = Output()

def update_filter(*args):
    global filtered_signal_global
    
    with output_plot:
        clear_output(wait=True)
        
        # 应用滤波
        if filter_type.value == '低通滤波':
            fft_filtered = apply_filter(fft_original, frequencies, '低通滤波', cutoff=cutoff_freq.value)
            title_filter = f'低通滤波 (截止: {cutoff_freq.value} Hz)'
        elif filter_type.value == '高通滤波':
            fft_filtered = apply_filter(fft_original, frequencies, '高通滤波', cutoff=cutoff_freq.value)
            title_filter = f'高通滤波 (截止: {cutoff_freq.value} Hz)'
        else:  # 带通滤波
            low = min(low_cutoff.value, high_cutoff.value)
            high = max(low_cutoff.value, high_cutoff.value)
            fft_filtered = apply_filter(fft_original, frequencies, '带通滤波', low_cut=low, high_cut=high)
            title_filter = f'带通滤波 ({low}-{high} Hz)'
        
        # 逆变换得到滤波后信号
        filtered_signal = np.real(np.fft.ifft(fft_filtered))
        filtered_signal_global = filtered_signal
        
        # 计算频谱幅度
        positive_mask = frequencies >= 0
        pos_freqs = frequencies[positive_mask]
        mag_original = np.abs(fft_original[positive_mask]) / N * 2
        mag_filtered = np.abs(fft_filtered[positive_mask]) / N * 2
        
        # 创建图形
        fig = plt.figure(figsize=(15, 10))
        
        # 1. 时域信号对比
        ax1 = plt.subplot(3, 1, 1)
        display_samples = int(time_range.value * sampling_freq)
        ax1.plot(t[:display_samples], original_signal[:display_samples], 'r-', alpha=0.5, label='原始信号', linewidth=1)
        ax1.plot(t[:display_samples], filtered_signal[:display_samples], 'b-', label='滤波后信号', linewidth=2)
        ax1.set_xlabel('时间 (秒)')
        ax1.set_ylabel('幅度')
        ax1.set_title('时域信号对比')
        ax1.legend()
        ax1.grid(True, alpha=0.3)
        
        # 2. 频谱对比
        ax2 = plt.subplot(3, 1, 2)
        ax2.plot(pos_freqs, mag_original, 'r-', alpha=0.5, label='原始频谱', linewidth=1)
        ax2.plot(pos_freqs, mag_filtered, 'b-', label='滤波后频谱', linewidth=2)
        ax2.set_xlabel('频率 (Hz)')
        ax2.set_ylabel('幅度')
        ax2.set_title(f'频域频谱对比 - {title_filter}')
        ax2.set_xlim(0, sampling_freq/2)
        ax2.legend()
        ax2.grid(True, alpha=0.3)
        
        # 标记滤波区域
        if filter_type.value == '低通滤波':
            ax2.axvspan(cutoff_freq.value, sampling_freq/2, color='red', alpha=0.1, label='滤除区域')
        elif filter_type.value == '高通滤波':
            ax2.axvspan(0, cutoff_freq.value, color='red', alpha=0.1, label='滤除区域')
        else:
            ax2.axvspan(0, low, color='red', alpha=0.1)
            ax2.axvspan(high, sampling_freq/2, color='red', alpha=0.1, label='滤除区域')
        ax2.legend()
        
        # 3. 滤波效果细节
        ax3 = plt.subplot(3, 1, 3)
        difference = original_signal - filtered_signal
        ax3.plot(t[:display_samples], difference[:display_samples], 'g-', label='差值 (被滤除的成分)', linewidth=1)
        ax3.set_xlabel('时间 (秒)')
        ax3.set_ylabel('幅度')
        ax3.set_title('被滤除的信号成分 (原始 - 滤波后)')
        ax3.legend()
        ax3.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # 打印统计信息
        print(f"\n{'='*60}")
        print(f"滤波类型: {title_filter}")
        print(f"原始信号能量: {np.sum(original_signal**2):.2f}")
        print(f"滤波后信号能量: {np.sum(filtered_signal**2):.2f}")
        print(f"能量保留比例: {np.sum(filtered_signal**2)/np.sum(original_signal**2)*100:.1f}%")
        print(f"{'='*60}")

In [ ]:
# 根据滤波类型显示/隐藏相应的滑块
def on_filter_type_change(change):
    if change['new'] == '带通滤波':
        cutoff_freq.layout.display = 'none'
        low_cutoff.layout.display = 'flex'
        high_cutoff.layout.display = 'flex'
    else:
        cutoff_freq.layout.display = 'flex'
        low_cutoff.layout.display = 'none'
        high_cutoff.layout.display = 'none'
    update_filter()

filter_type.observe(on_filter_type_change, names='value')
cutoff_freq.observe(update_filter, names='value')
low_cutoff.observe(update_filter, names='value')
high_cutoff.observe(update_filter, names='value')
time_range.observe(update_filter, names='value')

In [ ]:
# 导出数据功能
export_output = Output()

def export_data(b):
    global filtered_signal_global
    
    with export_output:
        clear_output(wait=True)
        
        if filtered_signal_global is None:
            print("请先调整滤波参数，生成滤波后信号！")
            return
        
        # 保存为 NumPy 格式
        np.save('filtered_signal.npy', filtered_signal_global)
        
        # 保存为文本格式（带时间轴）
        data_to_save = np.column_stack((t, filtered_signal_global))
        np.savetxt('filtered_signal.txt', data_to_save, header='时间(s) 幅度', fmt='%.6f')
        
        # 保存为 CSV 格式
        import csv
        with open('filtered_signal.csv', 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(['时间(s)', '幅度'])
            for time, amp in zip(t, filtered_signal_global):
                writer.writerow([f'{time:.6f}', f'{amp:.6f}'])
        
        print("\n✓ 数据导出成功！")
        print("\n导出的文件:")
        print("  - filtered_signal.npy  (NumPy 二进制格式)")
        print("  - filtered_signal.txt  (文本格式，带时间轴)")
        print("  - filtered_signal.csv  (CSV 格式，可直接用 Excel 打开)")
        print(f"\n数据统计:")
        print(f"  - 采样点数: {len(filtered_signal_global)}")
        print(f"  - 信号时长: {duration} 秒")
        print(f"  - 采样频率: {sampling_freq} Hz")
        print(f"  - 信号均值: {np.mean(filtered_signal_global):.6f}")
        print(f"  - 信号标准差: {np.std(filtered_signal_global):.6f}")

export_button = Button(
    description='导出滤波后数据',
    button_style='success',
    icon='download',
    layout={'width': '200px'}
)
export_button.on_click(export_data)

In [ ]:
# 重置参数功能
def reset_params(b):
    filter_type.value = '低通滤波'
    cutoff_freq.value = 50
    low_cutoff.value = 20
    high_cutoff.value = 80
    time_range.value = 0.5

reset_button = Button(
    description='重置参数',
    button_style='warning',
    icon='refresh',
    layout={'width': '150px'}
)
reset_button.on_click(reset_params)

## 3. 交互式滤波演示

### 操作说明

1. **选择滤波类型**：
   - **低通滤波**：保留低于截止频率的成分，平滑信号、去噪
   - **高通滤波**：保留高于截止频率的成分，提取细节、边缘
   - **带通滤波**：保留指定频率范围内的成分

2. **调整参数**：拖动滑块实时观察滤波效果

3. **导出数据**：点击按钮将滤波后信号保存为多种格式

---

**原始信号包含的频率成分**：5 Hz、15 Hz、30 Hz、60 Hz、120 Hz

In [ ]:
# 显示所有控件
control_box = VBox([
    filter_type,
    cutoff_freq,
    low_cutoff,
    high_cutoff,
    time_range,
    HBox([export_button, reset_button], spacing='20px'),
    export_output
])

# 初始隐藏带通滤波控件
low_cutoff.layout.display = 'none'
high_cutoff.layout.display = 'none'

display(control_box)
display(output_plot)

# 初始显示
update_filter()

## 4. 滤波效果分析示例

尝试以下参数组合，观察不同的滤波效果：

### 低通滤波示例
- **截止频率 10 Hz**：只保留 5 Hz 成分，信号非常平滑
- **截止频率 40 Hz**：保留 5、15、30 Hz 成分，去除 60、120 Hz
- **截止频率 100 Hz**：保留大部分成分，只去除最高的 120 Hz

### 高通滤波示例
- **截止频率 100 Hz**：只保留 120 Hz 成分，高频细节
- **截止频率 50 Hz**：保留 60、120 Hz 成分

### 带通滤波示例
- **20-40 Hz**：只保留 30 Hz 附近成分
- **10-70 Hz**：保留 15、30、60 Hz 成分

## 5. 加载导出的数据（可选）

In [ ]:
# 如果你导出了数据，可以用下面的代码加载验证
try:
    loaded_data = np.load('filtered_signal.npy')
    print(f"成功加载 filtered_signal.npy")
    print(f"数据形状: {loaded_data.shape}")
    print(f"数据范围: [{np.min(loaded_data):.4f}, {np.max(loaded_data):.4f}]")
except FileNotFoundError:
    print("请先导出数据，然后运行此单元格")

## 总结

本 Notebook 展示了：

1. **交互式频域滤波**：使用 ipywidgets 实现实时参数调整
2. **三种滤波类型**：低通、高通、带通滤波
3. **可视化对比**：
   - 时域信号对比（原始 vs 滤波后）
   - 频域频谱对比（显示滤除区域）
   - 差值信号（被滤除的成分）
4. **数据导出功能**：支持 .npy、.txt、.csv 三种格式
5. **能量分析**：显示滤波前后的能量变化

频域滤波是信号处理的强大工具，可以精确地去除或保留特定频率成分！